# T07 - 弘则信用债指数重构

## 项目概述

弘则信用债指数项目是一个完整的债券指数构建和分析系统，专注于构建信用债和金融债的标准10-100分位数指数。通过收益率分组和余额加权的方式计算指数，提供债券市场的分位数收益表现分析。

### 功能描述
- 构建信用债和金融债的分位数指数（10个分位数组）
- 计算全价和净价指数
- 支持每日更新和历史数据补充
- 提供交互式可视化图表

### 数据源
- **PostgreSQL (tsdb)**: 收益率曲线数据、债券基础信息
- **MySQL (bond)**: 市场行情数据、指数存储

### 输出结果
- 40个指数（信用债10分位 x 2价格类型 + 金融债10分位 x 2价格类型）
- 交互式HTML图表
- CSV数据文件

---

## 1. 环境配置

### 导入依赖

In [ ]:
# 基础库
import pandas as pd
import numpy as np
import warnings
import os
import gc
from datetime import datetime, timedelta
from tqdm import tqdm

warnings.filterwarnings('ignore')

# 数据库连接
import sqlalchemy
from sqlalchemy import text
import psycopg2

# 可视化
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.offline as pyo

# 缓存
import pyarrow.parquet as pq
import pyarrow as pa

print("Environment configured successfully!")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

### 配置参数

In [ ]:
from config import (
    PG_CONFIG, 
    MYSQL_CONFIG, 
    INDEX_CONFIG, 
    CACHE_DIR,
    OUTPUT_DIR
)

# 显示配置
print("=== Database Configuration ===")
print(f"PostgreSQL Host: {PG_CONFIG['host']}")
print(f"PostgreSQL Database: {PG_CONFIG['database']}")
print(f"MySQL Host: {MYSQL_CONFIG['host']}")
print(f"MySQL Database: {MYSQL_CONFIG['database']}")

print("\n=== Index Configuration ===")
print(f"Target Term: {INDEX_CONFIG['target_term']} years")
print(f"Percentile Groups: {INDEX_CONFIG['percentile_groups']}")
print(f"Bond Types: {INDEX_CONFIG['bond_types']}")
print(f"Price Types: {INDEX_CONFIG['price_types']}")

print("\n=== Directory Configuration ===")
print(f"Cache Directory: {CACHE_DIR}")
print(f"Output Directory: {OUTPUT_DIR}")

---

## 2. 数据获取

### 数据库连接

In [ ]:
class DatabaseManager:
    """数据库连接管理器"""
    
    def __init__(self):
        self.pg_conn = None
        self.sql_engine = None
        
    def connect_postgresql(self):
        """连接PostgreSQL数据库"""
        try:
            self.pg_conn = psycopg2.connect(
                host=PG_CONFIG['host'],
                port=PG_CONFIG['port'],
                user=PG_CONFIG['user'],
                password=PG_CONFIG['password'],
                database=PG_CONFIG['database']
            )
            print("PostgreSQL connected successfully!")
            return True
        except Exception as e:
            print(f"PostgreSQL connection failed: {e}")
            return False
    
    def connect_mysql(self):
        """连接MySQL数据库"""
        try:
            self.sql_engine = sqlalchemy.create_engine(
                f"mysql+pymysql://{MYSQL_CONFIG['user']}:{MYSQL_CONFIG['password']}@{MYSQL_CONFIG['host']}:{MYSQL_CONFIG['port']}/{MYSQL_CONFIG['database']}",
                poolclass=sqlalchemy.pool.NullPool,
                pool_pre_ping=True,
                pool_recycle=3600,
                connect_args={
                    'connect_timeout': 60,
                    'read_timeout': 60,
                    'write_timeout': 60
                }
            )
            print("MySQL connected successfully!")
            return True
        except Exception as e:
            print(f"MySQL connection failed: {e}")
            return False
    
    def close_connections(self):
        """关闭所有数据库连接"""
        if self.pg_conn:
            self.pg_conn.close()
            print("PostgreSQL connection closed.")
        if self.sql_engine:
            self.sql_engine.dispose()
            print("MySQL connection closed.")

# 创建数据库管理器实例
db = DatabaseManager()
db.connect_postgresql()
db.connect_mysql()

### 获取债券分类信息

In [ ]:
def get_bond_classification(pg_conn, force_refresh=False, cache_dir=CACHE_DIR):
    """
    获取债券分类信息
    
    Parameters:
    -----------
    pg_conn : psycopg2 connection
        PostgreSQL数据库连接
    force_refresh : bool
        是否强制刷新缓存
    cache_dir : str
        缓存目录
        
    Returns:
    --------
    tuple : (credit_bonds_list, finance_bonds_list)
    """
    cache_path = os.path.join(cache_dir, 'bond_classification.parquet')
    
    # 检查缓存
    if not force_refresh and os.path.exists(cache_path):
        print("Loading bond classification from cache...")
        cached_data = pd.read_parquet(cache_path)
        credit_bonds = cached_data[cached_data['bond_type'] == 'credit']['trade_code'].tolist()
        finance_bonds = cached_data[cached_data['bond_type'] == 'finance']['trade_code'].tolist()
        return credit_bonds, finance_bonds
    
    print("Getting bond classification from database...")
    
    # 获取信用债列表
    credit_query = """
    SELECT trade_code
    FROM basicinfo_credit
    WHERE ths_is_issuing_failure_bond != '是'
    """
    
    # 获取金融债列表
    finance_query = """
    SELECT trade_code
    FROM basicinfo_finance
    WHERE ths_is_issuing_failure_bond != '是'
    """
    
    credit_bonds = pd.read_sql(credit_query, pg_conn)
    finance_bonds = pd.read_sql(finance_query, pg_conn)
    
    # 保存缓存
    credit_bonds['bond_type'] = 'credit'
    finance_bonds['bond_type'] = 'finance'
    all_bonds = pd.concat([credit_bonds, finance_bonds], ignore_index=True)
    
    os.makedirs(cache_dir, exist_ok=True)
    all_bonds.to_parquet(cache_path, index=False)
    print(f"Bond classification cached: {len(credit_bonds)} credit bonds, {len(finance_bonds)} finance bonds")
    
    return credit_bonds['trade_code'].tolist(), finance_bonds['trade_code'].tolist()

# 获取债券分类
credit_bonds, finance_bonds = get_bond_classification(db.pg_conn)
print(f"\nTotal credit bonds: {len(credit_bonds)}")
print(f"Total finance bonds: {len(finance_bonds)}")

### 获取收益率数据

In [ ]:
def get_yield_data(pg_conn, start_date, end_date, credit_bonds, finance_bonds, 
                   target_term=3, cache_dir=CACHE_DIR, force_refresh=False):
    """
    获取收益率数据
    
    Parameters:
    -----------
    pg_conn : psycopg2 connection
        PostgreSQL数据库连接
    start_date : str
        开始日期 (YYYY-MM-DD)
    end_date : str
        结束日期 (YYYY-MM-DD)
    credit_bonds : list
        信用债代码列表
    finance_bonds : list
        金融债代码列表
    target_term : int
        目标期限（年）
    cache_dir : str
        缓存目录
    force_refresh : bool
        是否强制刷新缓存
        
    Returns:
    --------
    pd.DataFrame : 收益率数据
    """
    cache_filename = f"yield_data_{start_date}_{end_date}.parquet"
    cache_path = os.path.join(cache_dir, 'yield_data', cache_filename)
    
    os.makedirs(os.path.dirname(cache_path), exist_ok=True)
    
    # 检查缓存
    if not force_refresh and os.path.exists(cache_path):
        print(f"Loading yield data from cache: {cache_path}")
        return pd.read_parquet(cache_path)
    
    print(f"Getting yield data from {start_date} to {end_date}...")
    
    query = f"""
    SELECT dt, trade_code, stdyield, balance
    FROM hzcurve_credit
    WHERE dt >= '{start_date}' AND dt <= '{end_date}'
    AND target_term = {target_term}
    AND stdyield IS NOT NULL
    ORDER BY dt, stdyield DESC
    """
    
    yield_data = pd.read_sql(query, pg_conn)
    
    # 标记债券类型
    yield_data['bond_type'] = 'unknown'
    yield_data.loc[yield_data['trade_code'].isin(credit_bonds), 'bond_type'] = 'credit'
    yield_data.loc[yield_data['trade_code'].isin(finance_bonds), 'bond_type'] = 'finance'
    
    # 移除未分类的债券
    yield_data = yield_data[yield_data['bond_type'] != 'unknown']
    
    # 优化数据类型
    yield_data['dt'] = pd.to_datetime(yield_data['dt'])
    yield_data['stdyield'] = yield_data['stdyield'].astype('float32')
    yield_data['balance'] = yield_data['balance'].astype('float64')
    yield_data['bond_type'] = yield_data['bond_type'].astype('category')
    
    # 保存缓存
    yield_data.to_parquet(cache_path, index=False)
    print(f"Yield data cached: {len(yield_data)} records")
    
    return yield_data

# 设置日期范围
START_DATE = '2024-08-01'
END_DATE = '2025-02-14'

# 获取收益率数据
yield_data = get_yield_data(db.pg_conn, START_DATE, END_DATE, credit_bonds, finance_bonds)
print(f"\nYield data shape: {yield_data.shape}")
print(f"Date range: {yield_data['dt'].min()} to {yield_data['dt'].max()}")
print(f"\nSample data:")
yield_data.head()

### 获取价格数据

In [ ]:
def get_price_data(sql_engine, start_date, end_date, bond_type, cache_dir=CACHE_DIR, force_refresh=False):
    """
    获取价格数据
    
    Parameters:
    -----------
    sql_engine : sqlalchemy engine
        MySQL数据库引擎
    start_date : str
        开始日期 (YYYY-MM-DD)
    end_date : str
        结束日期 (YYYY-MM-DD)
    bond_type : str
        债券类型 ('credit' 或 'finance')
    cache_dir : str
        缓存目录
    force_refresh : bool
        是否强制刷新缓存
        
    Returns:
    --------
    pd.DataFrame : 价格数据
    """
    cache_filename = f"price_data_{bond_type}_{start_date}_{end_date}.parquet"
    cache_path = os.path.join(cache_dir, 'price_data', cache_filename)
    
    os.makedirs(os.path.dirname(cache_path), exist_ok=True)
    
    # 检查缓存
    if not force_refresh and os.path.exists(cache_path):
        print(f"Loading {bond_type} price data from cache...")
        return pd.read_parquet(cache_path)
    
    print(f"Fetching {bond_type} price data from {start_date} to {end_date}...")
    
    table_name = 'marketinfo_credit' if bond_type == 'credit' else 'marketinfo_finance'
    
    query = f"""
    SELECT DT, TRADE_CODE, ths_bond_balance_bond, 
           ths_valuate_full_price_cb_bond, ths_evaluate_net_price_cb_bond
    FROM {table_name}
    WHERE DT >= '{start_date}' AND DT <= '{end_date}'
    AND ths_bond_balance_bond IS NOT NULL
    AND ths_valuate_full_price_cb_bond IS NOT NULL
    AND ths_evaluate_net_price_cb_bond IS NOT NULL
    ORDER BY DT, TRADE_CODE
    """
    
    price_data = pd.read_sql(query, sql_engine)
    
    if not price_data.empty:
        price_data.columns = ['dt', 'trade_code', 'balance', 'full_price', 'net_price']
        
        # 优化数据类型
        price_data['dt'] = pd.to_datetime(price_data['dt'])
        price_data['balance'] = price_data['balance'].astype('float64')
        price_data['full_price'] = price_data['full_price'].astype('float32')
        price_data['net_price'] = price_data['net_price'].astype('float32')
        
        # 保存缓存
        price_data.to_parquet(cache_path, index=False)
        print(f"{bond_type.capitalize()} price data cached: {len(price_data)} records")
    else:
        print(f"No {bond_type} price data found!")
    
    return price_data

# 获取价格数据
price_data_credit = get_price_data(db.sql_engine, START_DATE, END_DATE, 'credit')
price_data_finance = get_price_data(db.sql_engine, START_DATE, END_DATE, 'finance')

print(f"\nCredit price data shape: {price_data_credit.shape}")
print(f"Finance price data shape: {price_data_finance.shape}")

---

## 3. 数据处理

### 分位数分组

In [ ]:
def create_percentile_groups(yield_data, cache_dir=CACHE_DIR, force_refresh=False):
    """
    创建分位数分组
    
    Parameters:
    -----------
    yield_data : pd.DataFrame
        收益率数据
    cache_dir : str
        缓存目录
    force_refresh : bool
        是否强制刷新缓存
        
    Returns:
    --------
    pd.DataFrame : 分组数据
    """
    os.makedirs(os.path.join(cache_dir, 'grouped_data'), exist_ok=True)
    
    print("Creating percentile groups...")
    
    unique_dates = sorted(yield_data['dt'].unique())
    all_grouped_data = []
    
    for date in tqdm(unique_dates, desc="Processing dates"):
        date_str = date.strftime('%Y-%m-%d')
        cache_path = os.path.join(cache_dir, 'grouped_data', f'grouped_{date_str}.parquet')
        
        # 检查缓存
        if not force_refresh and os.path.exists(cache_path):
            grouped_data = pd.read_parquet(cache_path)
        else:
            daily_data = yield_data[yield_data['dt'] == date]
            grouped_data = create_percentile_groups_for_date(daily_data, date)
            
            if not grouped_data.empty:
                grouped_data.to_parquet(cache_path, index=False)
        
        if not grouped_data.empty:
            all_grouped_data.append(grouped_data)
    
    if all_grouped_data:
        combined_data = pd.concat(all_grouped_data, ignore_index=True)
        print(f"Total grouped records: {len(combined_data)}")
        return combined_data
    else:
        return pd.DataFrame()


def create_percentile_groups_for_date(daily_data, date):
    """
    为单个日期创建分位数分组
    
    Parameters:
    -----------
    daily_data : pd.DataFrame
        当日收益率数据
    date : datetime
        日期
        
    Returns:
    --------
    pd.DataFrame : 分组数据
    """
    grouped_data = []
    
    for bond_type in ['credit', 'finance']:
        type_data = daily_data[daily_data['bond_type'] == bond_type].copy()
        
        if len(type_data) == 0:
            continue
        
        # 按收益率升序排列
        type_data = type_data.sort_values('stdyield', ascending=True)
        
        # 计算分位数
        n_bonds = len(type_data)
        
        for i in range(10):
            start_pct = i * 10
            end_pct = (i + 1) * 10
            
            start_idx = int(n_bonds * start_pct / 100)
            end_idx = int(n_bonds * end_pct / 100)
            
            if start_idx >= n_bonds:
                continue
            if end_idx > n_bonds:
                end_idx = n_bonds
            
            group_data = type_data.iloc[start_idx:end_idx]
            
            if len(group_data) > 0:
                for _, row in group_data.iterrows():
                    grouped_data.append({
                        'dt': date,
                        'trade_code': row['trade_code'],
                        'bond_type': bond_type,
                        'percentile_group': f"{start_pct}-{end_pct}",
                        'stdyield': row['stdyield'],
                        'balance': row['balance']
                    })
    
    if grouped_data:
        result_df = pd.DataFrame(grouped_data)
        result_df['bond_type'] = result_df['bond_type'].astype('category')
        result_df['percentile_group'] = result_df['percentile_group'].astype('category')
        result_df['stdyield'] = result_df['stdyield'].astype('float32')
        result_df['balance'] = result_df['balance'].astype('float64')
        return result_df
    else:
        return pd.DataFrame()

# 创建分位数分组
grouped_data = create_percentile_groups(yield_data)
print(f"\nGrouped data shape: {grouped_data.shape}")
print(f"\nPercentile group distribution:")
print(grouped_data['percentile_group'].value_counts().sort_index())

---

## 4. 核心逻辑

### 指数计算

In [ ]:
def calculate_indices(grouped_data, price_data_dict, cache_dir=CACHE_DIR, force_refresh=False):
    """
    计算指数
    
    Parameters:
    -----------
    grouped_data : pd.DataFrame
        分组数据
    price_data_dict : dict
        价格数据字典 {'credit': df, 'finance': df}
    cache_dir : str
        缓存目录
    force_refresh : bool
        是否强制刷新缓存
        
    Returns:
    --------
    pd.DataFrame : 指数数据
    """
    os.makedirs(os.path.join(cache_dir, 'indices_data'), exist_ok=True)
    
    print("Calculating indices...")
    
    unique_dates = sorted(grouped_data['dt'].unique())
    all_indices = []
    
    for date in tqdm(unique_dates, desc="Calculating indices"):
        date_str = date.strftime('%Y-%m-%d')
        cache_path = os.path.join(cache_dir, 'indices_data', f'indices_{date_str}.parquet')
        
        # 检查缓存
        if not force_refresh and os.path.exists(cache_path):
            indices_data = pd.read_parquet(cache_path)
        else:
            indices_data = calculate_indices_for_date(date, grouped_data, price_data_dict)
            
            if not indices_data.empty:
                indices_data.to_parquet(cache_path, index=False)
        
        if not indices_data.empty:
            all_indices.append(indices_data)
    
    if all_indices:
        combined_data = pd.concat(all_indices, ignore_index=True)
        print(f"Total index points calculated: {len(combined_data)}")
        return combined_data
    else:
        return pd.DataFrame()


def calculate_indices_for_date(date, grouped_data, price_data_dict):
    """
    计算单个日期的指数
    
    Parameters:
    -----------
    date : datetime
        日期
    grouped_data : pd.DataFrame
        分组数据
    price_data_dict : dict
        价格数据字典
        
    Returns:
    --------
    pd.DataFrame : 指数数据
    """
    daily_grouped = grouped_data[grouped_data['dt'] == date]
    if daily_grouped.empty:
        return pd.DataFrame()
    
    all_indices = []
    
    for bond_type in ['credit', 'finance']:
        type_data = daily_grouped[daily_grouped['bond_type'] == bond_type]
        if type_data.empty:
            continue
        
        # 获取当日价格数据
        if bond_type not in price_data_dict or price_data_dict[bond_type].empty:
            continue
        
        daily_price_data = price_data_dict[bond_type][
            price_data_dict[bond_type]['dt'] == date
        ]
        
        if daily_price_data.empty:
            continue
        
        # 合并数据
        merged_data = type_data.merge(
            daily_price_data, on=['dt', 'trade_code'], how='inner', suffixes=('_yield', '_price')
        )
        
        if merged_data.empty:
            continue
        
        # 使用价格数据中的balance
        merged_data['balance'] = merged_data['balance_price']
        
        for percentile_group in merged_data['percentile_group'].unique():
            group_data = merged_data[merged_data['percentile_group'] == percentile_group]
            
            if group_data.empty or group_data['balance'].sum() == 0:
                continue
            
            # 计算加权平均
            weights = group_data['balance'] / group_data['balance'].sum()
            
            full_price_index = (group_data['full_price'] * weights).sum()
            net_price_index = (group_data['net_price'] * weights).sum()
            
            all_indices.extend([
                {
                    'dt': date,
                    'index_code': f"{bond_type.title()}_{percentile_group}_Full",
                    'index_value': full_price_index,
                    'index_type': 'full_price'
                },
                {
                    'dt': date,
                    'index_code': f"{bond_type.title()}_{percentile_group}_Net",
                    'index_value': net_price_index,
                    'index_type': 'net_price'
                }
            ])
    
    if all_indices:
        result_df = pd.DataFrame(all_indices)
        result_df['index_type'] = result_df['index_type'].astype('category')
        result_df['index_value'] = result_df['index_value'].astype('float32')
        return result_df
    else:
        return pd.DataFrame()

# 准备价格数据字典
price_data_dict = {
    'credit': price_data_credit,
    'finance': price_data_finance
}

# 计算指数
indices_data = calculate_indices(grouped_data, price_data_dict)
print(f"\nIndices data shape: {indices_data.shape}")
print(f"\nIndex codes:")
for code in sorted(indices_data['index_code'].unique()):
    print(f"  {code}")

---

## 5. 执行与测试

### 数据验证

In [ ]:
# 数据验证
print("=== Data Validation ===")
print(f"\n1. Yield Data:")
print(f"   - Records: {len(yield_data)}")
print(f"   - Date range: {yield_data['dt'].min()} to {yield_data['dt'].max()}")
print(f"   - Credit bonds: {len(yield_data[yield_data['bond_type'] == 'credit'])}")
print(f"   - Finance bonds: {len(yield_data[yield_data['bond_type'] == 'finance'])}")

print(f"\n2. Grouped Data:")
print(f"   - Records: {len(grouped_data)}")
print(f"   - Unique dates: {grouped_data['dt'].nunique()}")

print(f"\n3. Indices Data:")
print(f"   - Records: {len(indices_data)}")
print(f"   - Unique index codes: {indices_data['index_code'].nunique()}")
print(f"   - Date range: {indices_data['dt'].min()} to {indices_data['dt'].max()}")

# 显示指数统计
print(f"\n=== Index Statistics ===")
index_stats = indices_data.groupby('index_code')['index_value'].agg(['mean', 'std', 'min', 'max'])
index_stats

### 生成可视化图表

In [ ]:
def create_interactive_chart(indices_data, output_file='bond_indices_dashboard.html'):
    """
    创建交互式图表
    
    Parameters:
    -----------
    indices_data : pd.DataFrame
        指数数据
    output_file : str
        输出文件路径
        
    Returns:
    --------
    plotly Figure : 图表对象
    """
    print("Creating interactive chart...")
    
    # 转换日期格式
    indices_data['dt'] = pd.to_datetime(indices_data['dt'])
    
    # 创建子图
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=('Credit Bond Indices - Full Price', 'Credit Bond Indices - Net Price',
                      'Finance Bond Indices - Full Price', 'Finance Bond Indices - Net Price'),
        vertical_spacing=0.1,
        horizontal_spacing=0.1
    )
    
    # 定义颜色
    colors = px.colors.qualitative.Set3
    
    # 绘制信用债全价指数
    credit_full = indices_data[
        (indices_data['index_code'].str.contains('Credit')) & 
        (indices_data['index_type'] == 'full_price')
    ]
    
    for i, index_code in enumerate(credit_full['index_code'].unique()):
        data = credit_full[credit_full['index_code'] == index_code]
        fig.add_trace(
            go.Scatter(
                x=data['dt'], 
                y=data['index_value'],
                mode='lines',
                name=index_code,
                line=dict(color=colors[i % len(colors)]),
                hovertemplate='<b>%{fullData.name}</b><br>Date: %{x}<br>Value: %{y:.4f}<extra></extra>'
            ),
            row=1, col=1
        )
    
    # 绘制信用债净价指数
    credit_net = indices_data[
        (indices_data['index_code'].str.contains('Credit')) & 
        (indices_data['index_type'] == 'net_price')
    ]
    
    for i, index_code in enumerate(credit_net['index_code'].unique()):
        data = credit_net[credit_net['index_code'] == index_code]
        fig.add_trace(
            go.Scatter(
                x=data['dt'], 
                y=data['index_value'],
                mode='lines',
                name=index_code,
                line=dict(color=colors[i % len(colors)]),
                hovertemplate='<b>%{fullData.name}</b><br>Date: %{x}<br>Value: %{y:.4f}<extra></extra>',
                showlegend=False
            ),
            row=1, col=2
        )
    
    # 绘制金融债全价指数
    finance_full = indices_data[
        (indices_data['index_code'].str.contains('Finance')) & 
        (indices_data['index_type'] == 'full_price')
    ]
    
    for i, index_code in enumerate(finance_full['index_code'].unique()):
        data = finance_full[finance_full['index_code'] == index_code]
        fig.add_trace(
            go.Scatter(
                x=data['dt'], 
                y=data['index_value'],
                mode='lines',
                name=index_code,
                line=dict(color=colors[i % len(colors)]),
                hovertemplate='<b>%{fullData.name}</b><br>Date: %{x}<br>Value: %{y:.4f}<extra></extra>',
                showlegend=False
            ),
            row=2, col=1
        )
    
    # 绘制金融债净价指数
    finance_net = indices_data[
        (indices_data['index_code'].str.contains('Finance')) & 
        (indices_data['index_type'] == 'net_price')
    ]
    
    for i, index_code in enumerate(finance_net['index_code'].unique()):
        data = finance_net[finance_net['index_code'] == index_code]
        fig.add_trace(
            go.Scatter(
                x=data['dt'], 
                y=data['index_value'],
                mode='lines',
                name=index_code,
                line=dict(color=colors[i % len(colors)]),
                hovertemplate='<b>%{fullData.name}</b><br>Date: %{x}<br>Value: %{y:.4f}<extra></extra>',
                showlegend=False
            ),
            row=2, col=2
        )
    
    # 更新布局
    fig.update_layout(
        title_text="Bond Indices Dashboard",
        title_x=0.5,
        height=800,
        showlegend=True,
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="center",
            x=0.5
        ),
        hovermode='x unified'
    )
    
    # 添加范围选择器
    for i in range(1, 3):
        for j in range(1, 3):
            fig.update_xaxes(
                rangeselector=dict(
                    buttons=list([
                        dict(count=1, label="1m", step="month", stepmode="backward"),
                        dict(count=3, label="3m", step="month", stepmode="backward"),
                        dict(count=6, label="6m", step="month", stepmode="backward"),
                        dict(count=1, label="1y", step="year", stepmode="backward"),
                        dict(step="all")
                    ])
                ),
                type="date",
                row=i, col=j
            )
    
    # 保存HTML文件
    pyo.plot(fig, filename=output_file, auto_open=False)
    print(f"Interactive chart saved as {output_file}")
    
    return fig

# 创建图表
output_path = os.path.join(OUTPUT_DIR, 'bond_indices_dashboard.html')
fig = create_interactive_chart(indices_data.copy(), output_path)

# 显示图表
fig.show()

### 保存数据

In [ ]:
# 保存最终结果
print("Saving final results...")

# 保存为CSV
indices_csv_path = os.path.join(OUTPUT_DIR, 'bond_indices_data.csv')
indices_data.to_csv(indices_csv_path, index=False)
print(f"Indices data saved to: {indices_csv_path}")

grouped_csv_path = os.path.join(OUTPUT_DIR, 'bond_grouped_data.csv')
grouped_data.to_csv(grouped_csv_path, index=False)
print(f"Grouped data saved to: {grouped_csv_path}")

# 保存为Parquet
indices_parquet_path = os.path.join(CACHE_DIR, 'final_indices_data.parquet')
indices_data.to_parquet(indices_parquet_path, index=False)
print(f"Indices data cached to: {indices_parquet_path}")

grouped_parquet_path = os.path.join(CACHE_DIR, 'final_grouped_data.parquet')
grouped_data.to_parquet(grouped_parquet_path, index=False)
print(f"Grouped data cached to: {grouped_parquet_path}")

print("\nAll results saved successfully!")

---

## 6. 工具函数

### 每日更新函数

In [ ]:
def update_daily_indices(target_date=None, db=None, credit_bonds=None, finance_bonds=None):
    """
    更新日指数
    
    Parameters:
    -----------
    target_date : str or date
        目标日期，默认为昨天
    db : DatabaseManager
        数据库管理器
    credit_bonds : list
        信用债代码列表
    finance_bonds : list
        金融债代码列表
        
    Returns:
    --------
    bool : 是否成功
    """
    if target_date is None:
        target_date = datetime.now().date() - timedelta(days=1)
    
    if isinstance(target_date, str):
        target_date = datetime.strptime(target_date, '%Y-%m-%d').date()
    
    date_str = target_date.strftime('%Y-%m-%d')
    print(f"Updating indices for {date_str}...")
    
    try:
        # 获取收益率数据
        yield_data = get_yield_data(
            db.pg_conn, date_str, date_str, credit_bonds, finance_bonds
        )
        
        if yield_data.empty:
            print(f"No yield data available for {date_str}")
            return False
        
        # 创建分位数分组
        grouped_data = create_percentile_groups_for_date(
            yield_data[yield_data['dt'] == pd.Timestamp(date_str)], 
            pd.Timestamp(date_str)
        )
        
        if grouped_data.empty:
            print(f"No grouped data created for {date_str}")
            return False
        
        # 获取价格数据
        price_credit = get_price_data(db.sql_engine, date_str, date_str, 'credit')
        price_finance = get_price_data(db.sql_engine, date_str, date_str, 'finance')
        
        price_data_dict = {
            'credit': price_credit,
            'finance': price_finance
        }
        
        # 计算指数
        indices_data = calculate_indices_for_date(
            pd.Timestamp(date_str), grouped_data, price_data_dict
        )
        
        if indices_data.empty:
            print(f"No indices calculated for {date_str}")
            return False
        
        print(f"Successfully calculated {len(indices_data)} indices for {date_str}")
        return True
        
    except Exception as e:
        print(f"Error updating indices for {date_str}: {e}")
        return False

# 示例：更新昨天的数据
# update_daily_indices(db=db, credit_bonds=credit_bonds, finance_bonds=finance_bonds)

### 历史数据补充函数

In [ ]:
def backfill_historical_indices(start_date, end_date, db=None, credit_bonds=None, finance_bonds=None):
    """
    补充历史指数数据
    
    Parameters:
    -----------
    start_date : str
        开始日期 (YYYY-MM-DD)
    end_date : str
        结束日期 (YYYY-MM-DD)
    db : DatabaseManager
        数据库管理器
    credit_bonds : list
        信用债代码列表
    finance_bonds : list
        金融债代码列表
        
    Returns:
    --------
    dict : 处理结果统计
    """
    print(f"Starting historical index backfill from {start_date} to {end_date}...")
    
    # 生成日期列表
    date_list = pd.date_range(start_date, end_date, freq='D')
    
    success_count = 0
    error_count = 0
    
    for date in tqdm(date_list, desc="Processing dates"):
        # 跳过周末
        if date.dayofweek >= 5:
            continue
        
        try:
            if update_daily_indices(date.date(), db, credit_bonds, finance_bonds):
                success_count += 1
            else:
                error_count += 1
        except Exception as e:
            print(f"Error processing {date.strftime('%Y-%m-%d')}: {e}")
            error_count += 1
    
    result = {
        'success': success_count,
        'error': error_count,
        'total': success_count + error_count
    }
    
    print(f"\nBackfill completed:")
    print(f"  Successful: {success_count}")
    print(f"  Errors: {error_count}")
    
    return result

# 示例：补充历史数据
# backfill_historical_indices('2024-01-01', '2024-12-31', db, credit_bonds, finance_bonds)

### 关闭数据库连接

In [ ]:
# 关闭数据库连接
db.close_connections()
print("\nNotebook execution completed!")

---

## 总结

本Notebook完成了以下工作：

1. **数据获取**: 从PostgreSQL和MySQL数据库获取债券收益率和价格数据
2. **数据处理**: 按收益率分位数对债券进行分组
3. **指数计算**: 使用余额加权计算全价和净价指数
4. **可视化**: 生成交互式Plotly图表
5. **数据导出**: 保存为CSV和Parquet格式

### 输出文件
- `bond_indices_dashboard.html` - 交互式图表
- `bond_indices_data.csv` - 指数数据
- `bond_grouped_data.csv` - 分组数据

### 使用建议
- 使用 `update_daily_indices()` 函数进行每日更新
- 使用 `backfill_historical_indices()` 函数补充历史数据
- 缓存机制可加速重复运行